# 🤖 Construção e Comparação de Modelos de Machine Learning - Titanic

Este notebook apresenta a segunda etapa do meu projeto de aprendizado:
1. **Tratamento de Dados**: Preenchimento de dados nulos de maneira inteligente.
2. **Engenharia de Features**: Extração de títulos (*Mr, Mrs, Miss, Master*) e cálculo do tamanho de família.
3. **Treinamento de Modelos**: Comparando Regressão Logística, Árvore de Decisão, Random Forest e XGBoost.
4. **Validação Cruzada**: Garantindo que o modelo avalia com precisão o seu poder de generalização.
5. **Geração do Arquivo para o Kaggle**.

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.processamento import preparar_dados
from src.modelagem import treinar_e_comparar_modelos, gerar_grafico_importancia

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Carregando e Processando os Dados

In [ ]:
df_treino = pd.read_csv('../data/train.csv')
df_teste = pd.read_csv('../data/test.csv')

X_treino, y_treino, X_teste, ids_teste = preparar_dados(df_treino, df_teste)
print(f'Dimensão dos dados de treino após pré-processamento: {X_treino.shape}')
X_treino.head()

## 2. Comparação de Algoritmos com Validação Cruzada (5-Fold CV)

In [ ]:
df_resultados, modelos = treinar_e_comparar_modelos(X_treino, y_treino)
df_resultados

## 3. Analisando os Atributos Mais Importantes
O modelo Random Forest nos permite ver quais colunas tiveram mais peso na previsão.

In [ ]:
modelo_rf = modelos['Random Forest']
modelo_rf.fit(X_treino, y_treino)

importancias = pd.Series(modelo_rf.feature_importances_, index=X_treino.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
importancias.tail(10).plot(kind='barh', color='#2b5c8f', edgecolor='black', ax=ax)
ax.set_title('Importância das Variáveis (Random Forest)', fontsize=12, fontweight='bold')
plt.show()

## 4. Gerando Submissão Kaggle
Treinando com todos os dados para prever os passageiros do arquivo `test.csv`.

In [ ]:
predicoes = modelo_rf.predict(X_teste)

submissao = pd.DataFrame({
    'PassengerId': ids_teste,
    'Survived': predicoes
})
submissao.to_csv('../submissions/submission.csv', index=False)
print('✅ Submissão final gerada em ../submissions/submission.csv')
submissao.head()